# coAgent on Colab

An AI coding agent that runs entirely inside this notebook: your repo is cloned into
the VM, a 7B model is loaded locally, and every checkpoint is committed to a branch
you can review. Your code is never sent to a third-party API.

**Runtime → Change runtime type → T4 GPU** before you start. Free tier is the target.

---

### Before anything else: is this machine good enough?

In [ ]:
!pip -q install "co-agent-coder[colab] @ git+https://github.com/lyle-mlengineer/coagent"
# ^ the [colab] extra is required: it brings torch, transformers, and
#   bitsandbytes. Without it the 4-bit load fails with
#   "Using `bitsandbytes` 4-bit quantization requires bitsandbytes".

from coagent.llm import preflight_gpu

check = preflight_gpu()
print(check.describe())
print("verdict:", "OK — a free-tier T4" if check.ok else "REFUSED")
if not check.ok:
    print("\n" + check.reason)

If the verdict says **REFUSED**, stop here and fix it — usually *Runtime → Change
runtime type → T4 GPU*. coAgent refuses rather than downloading four gigabytes of
weights onto a runtime that cannot hold them.

### The agent: one cell, three lines

Replace the URL with your own repository. On a private repo, add a GitHub token in the
Colab **Secrets** panel (key icon, left edge) named `GITHUB_TOKEN` — it is used through
a `GIT_ASKPASS` helper and never written into the clone URL or the git config.

In [ ]:
from coagent import coAgent
agent = coAgent(repo="https://github.com/lyle-mlengineer/coagent-demo.git")
agent.chat("add a chunked CSV loader and test that dtypes are preserved")

That is the whole install story. What follows is what the three lines hide.

### What just happened

The run above is in your git history, on a branch named `coagent/<session-id>`. Nothing
was pushed anywhere: the branch is local until you ask.

```
git log --oneline coagent/<session>     # every step is a commit
git diff main...coagent/<session>       # one reviewable diff
```

In [ ]:
print(agent.status_line())
print(agent.show_log(last=5))

### If the runtime dies

It will. Colab reclaims VMs without warning, and that is the constraint the whole design
works backwards from. The repository is the memory, not the VM:

```python
agent = coAgent(repo="https://github.com/lyle-mlengineer/coagent-demo.git")
print(agent.resume())      # where the last session stopped, and what is left
```

### Approvals

Risky actions — installing packages, network calls, anything outside the plan — stop and
ask. Two buttons appear: use them, or type a command in a fresh cell if the widget state
was lost when the tab reconnected.

| type this | to |
| --- | --- |
| `agent.cell.command("approve")` | allow the pending action |
| `agent.cell.command("deny")` | refuse it |
| `agent.cell.command("stop")` | end the task, keeping completed steps committed |

Nothing is approved by silence: if no answer arrives within ten minutes, the action is
denied.

### Optional: Google Drive

Without Drive, artifacts larger than 5 MB are reported as a pointer rather than stored —
the server has no durable disk. Mounting Drive gives the run somewhere to put them.

```python
from google.colab import drive
drive.mount("/content/drive")
```

Run that **before** building the agent. Declining is fine; nothing breaks.

---

Docs: [`README.md`](https://github.com/lyle-mlengineer/coagent) ·
[`PRD.md`](https://github.com/lyle-mlengineer/coagent/blob/main/PRD.md) ·
[`DESIGN.md`](https://github.com/lyle-mlengineer/coagent/blob/main/DESIGN.md)

Apache 2.0.